# 02 / Brain-network communities — advanced challenge

**Question:** Can QAOA recover a useful division of a brain-like connectivity network into communities?

This notebook explains the network-science objective first, then compares **exact classical search**, a **classical heuristic** and **QAOA simulation**. Real IQM execution is an optional final step.

**Scientific scope:** the graph is **synthetic**, not empirical patient data. The experiment explores algorithms; it does not establish a neuroscience discovery or quantum advantage.

## 1 / What are brain networks and communities?

A **graph** represents a network:

- **Node:** a brain region (or, here, one of six fictional regions)
- **Edge:** a connection between two regions
- **Weight:** a number measuring the strength of that connection
- **Community:** a set of nodes more densely or strongly linked to each other than expected under a chosen model

Our data are **synthetic**, not measured brain data. We deliberately construct two groups, nodes **0–2** and **3–5**, with stronger connections within each group. The algorithm's job is to recover a useful grouping from the connectivity values—not from prior knowledge of how we generated them.

**Important:** This is a demonstration of an optimization method, not a claim about biological brain systems.

## 2 / What are we maximizing?

We want to assign the six nodes to **two communities**, maximizing **modularity** $Q$.

**Max-Cut** (beginner notebook) rewards edges **between** groups. **Modularity** instead rewards connections **within** groups when they are stronger than expected from a statistical reference called a **null model**.

| Term | Meaning |
|---|---|
| Connectivity matrix $A$ | Table: $A_{ij}$ is the observed weight between nodes $i$ and $j$ |
| Node strength $k_i$ | Sum of weights attached to node $i$ |
| Null model | Reference expected connectivity based on node strengths |
| Modularity matrix $B$ | $B_{ij}$ = observed weight minus expected weight |
| Modularity $Q$ | Score evaluating the overall division into communities |
| QAOA | Hybrid quantum–classical algorithm that searches for high-scoring assignments |

A high $Q$ means strong community structure **relative to the chosen null model**. It does not establish biological significance.

## 3 / The mathematical objective, step by step

For an undirected, nonnegative weighted graph:

1. **Observed connectivity** between nodes `i` and `j` is $A_{ij}$
2. **Node strength** is $k_i=\sum_j A_{ij}$
3. **Total network weight counted in both directions** is $2m=\sum_{ij}A_{ij}$
4. Under a weighted strength-preserving null model, the **expected** connection is $k_ik_j/(2m)$
5. A pair contributes to modularity only when both nodes have the **same community label** $c_i=c_j$


$$
Q=\frac{1}{2m}\sum_{i,j}\left(A_{ij}-\frac{k_i k_j}{2m}\right)\mathbf{1}[c_i=c_j]
$$


Here, $\mathbf{1}[c_i=c_j]$ is 1 when the two nodes share a community and 0 otherwise.

**Intuition:** If a pair of regions is more strongly connected than the null model predicts, placing them together tends to increase modularity. The normalization is crucial for comparing partitions of the *same* network.

## 4 / See the network and its matrices

Run the next cell to generate **one set of figures directly from the same synthetic data**.

- **Network diagram:** nodes 0–2 and 3–5 are colored as the two planted groups. Thick lines are stronger measured connections.
- **Left matrix $A$:** the darker the square, the **stronger the observed connection**. Values here are nonnegative; the diagonal is zero (no self-connections).
- **Right matrix $B$:** **red** means stronger than the null-model expectation; **blue** means weaker than expected. A blue entry **does not** mean a negative measured connection.

The two dark blocks in $A$ and two red blocks in $B$ suggest two communities. **These are input network descriptions, not QAOA results.**

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'qsip_quantum.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.colors import TwoSlopeNorm
from qsip_quantum import (
    brain_graph, draw_graph, modularity_matrix, binary_modularity,
    modularity_to_signed_cut, cut_score, weighted_edges, brute_force,
    qaoa_circuit, simulate_counts, best_from_counts, optimize_p1, bits_from_qiskit
)
N_NODES = 6  # Use 4 for a smaller hardware experiment; rerun from the top
G = brain_graph(n=N_NODES, seed=4)
n = G.number_of_nodes()
A, B, two_m = modularity_matrix(G)
planted_groups = tuple(0 if i < n // 2 else 1 for i in range(n))
fig, ax = draw_graph(G, planted_groups, title='Synthetic network — two planted groups')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.2))
img_a = axes[0].imshow(A, cmap='Greys', vmin=0, vmax=1)
img_b = axes[1].imshow(B, cmap='RdBu_r', norm=TwoSlopeNorm(vcenter=0))
for ax, title in zip(axes, ['A — measured connection strength', 'B — observed minus expected']):
    ax.set(title=title, xlabel='Node', ylabel='Node')
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
fig.colorbar(img_a, ax=axes[0], fraction=.046, pad=.04, label='Connection weight')
fig.colorbar(img_b, ax=axes[1], fraction=.046, pad=.04, label='Relative to expectation')
fig.suptitle('Same network, two mathematical descriptions', fontsize=14, y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
# Compare two possible community assignments (not quantum results)
all_together = tuple(0 for _ in range(n))
q_two = binary_modularity(planted_groups, B, two_m)
q_one = binary_modularity(all_together, B, two_m)
print(f'All nodes together: Q = {q_one:.4f}')
print(f'Two planted groups: Q = {q_two:.4f}')
print('Interpretation: the higher Q assigns communities that better match this network.')

## 5 / Find the best two-community assignment classically

We check every possible binary assignment. With six nodes, that means $2^6=64$ bitstrings (some represent the same partition with swapped group labels).

The **exact modularity** is the best $Q$ attainable when the network must be split into at most two groups. Compare the optimal assignment with the planted grouping. You can use this result as the ground truth for evaluating QAOA on this toy problem.

In [ ]:
(Q_exact, bits_exact), all_partitions = brute_force(
    n, lambda bits: binary_modularity(bits, B, two_m)
)
print('Exact two-community modularity:', round(Q_exact, 4))
print('One maximizing partition:', bits_exact)
fig, ax = draw_graph(G, bits_exact, title='Optimal two-community split')
plt.show()
print('This is the exact two-community baseline, not a quantum result.')

### Why does QUBO appear again?

We are still choosing a 0-or-1 label for each node. Those labels are the **decision variables**, but this time the **objective** is modularity, not the number of ordinary crossing edges.

- **QUBO**: any objective made of linear and pairwise products of binary choices. It describes the mathematical problem; it does not solve it.
- **Ising**: the same binary objective rewritten with $s_i=1-2x_i$, so $s_i$ is $-1$ or $+1$. It can be represented with Pauli-$Z$ operators in a quantum circuit.
- **Hamiltonian**: here, the quantum cost operator corresponding to our modularity-related score. It is not a direct measurement of biological energy.
- **QAOA**: the algorithm that uses cost and mixing operations to generate candidate assignments.

In the advanced notebook, *signed* edge weights account for the fact that modularity can penalize some node pairs being grouped or separated. **Ordinary positive-weight Max-Cut and modularity are different objectives**, even though both can be written as a quadratic binary problem.


## 6 / From modularity to QUBO and QAOA

Like Max-Cut, our assignment uses one binary variable $x_i\in\{0,1\}$ for each node. The **modularity matrix** is:

$$
B_{ij}=A_{ij}-\frac{k_i k_j}{2m}.
$$

For exactly two possible community labels, maximizing $Q$ is equivalent—apart from a constant independent of the assignment—to maximizing a **signed cut**:

$$
C_{\mathrm{signed}}(x)=\sum_{i<j}\left(-\frac{2B_{ij}}{2m}\right)(x_i+x_j-2x_ix_j).
$$

Unlike ordinary Max-Cut, effective weights may be **negative**. This is still a **QUBO** because it is quadratic in binary variables. We turn it into a qubit cost operator, then apply QAOA. The next cell checks that the signed-cut objective ranks every assignment in the same order as modularity.

In [ ]:
signed_edges = modularity_to_signed_cut(B, two_m)
weighted_best = cut_score(bits_exact, signed_edges)
print('Signed edges:', len(signed_edges), 'for', n, 'nodes')
print('Signed cut of classical optimum:', round(weighted_best, 5))
# Verify mathematically that the two objectives produce the same ordering.
checks = []
for (Q, bits) in all_partitions:
    checks.append(Q - cut_score(bits, signed_edges))
assert np.allclose(checks, checks[0], atol=1e-10)
print('Verified: Q = signed-cut + constant across all assignments.')

## 7 / Classical benchmark

A **greedy community-detection algorithm** repeatedly merges groups when doing so improves modularity. It is fast, but need not reach the global optimum.

Here it is included as a second classical reference. It may produce **more than two groups**, so note its community count before comparing it with the fixed two-community QAOA task.

In [ ]:
communities = list(nx.algorithms.community.greedy_modularity_communities(G, weight='weight'))
Q_greedy = nx.algorithms.community.modularity(G, communities, weight='weight')
print('Greedy modularity score:', round(Q_greedy, 4))
print('Greedy community count:', len(communities))
print('Exact two-community score:', round(Q_exact, 4))

## 8 / Run QAOA in the simulator

QAOA uses the signed quadratic objective from Section 6:

1. A **cost layer** encodes the objective as quantum phases.
2. A **mixer layer** changes the probability distribution over assignments.
3. A classical optimizer tunes the **adjustable rotation angles** $\gamma$ (cost) and $\beta$ (mixer) to increase **expected modularity**. This is what *tunable circuit* means.
4. The simulator returns **bitstrings**; each bit assigns one node to a community.

**What the result means:** The *best sampled modularity* is the best assignment seen in the simulated measurements. The *mean modularity* averages the results over all simulated shots. Neither is a measurement from IQM hardware.
**What runs where:** Angle tuning happens using an **ideal simulator**; optional IQM hardware submission below samples the resulting optimized circuit. It does not train the angles on IQM hardware.


In [ ]:
optimizer = optimize_p1(
    n, signed_edges,
    score_fn=lambda bits: binary_modularity(bits, B, two_m),
    maxiter=80
)
gamma, beta = optimizer.x
qaoa = qaoa_circuit(n, signed_edges, (gamma,), (beta,))
print('Optimized gamma, beta:', np.round([gamma, beta], 3))
print('Circuit depth before hardware transpilation:', qaoa.depth())
counts = simulate_counts(qaoa, shots=1024, seed=25)
Q_sample, label, number = best_from_counts(
    counts, lambda bits: binary_modularity(bits, B, two_m)
)
print('Best sampled modularity:', round(Q_sample, 4))
print('Exact two-community modularity:', round(Q_exact, 4))
print('Best sample:', label, 'observations:', number)
fig, ax = draw_graph(G, bits_from_qiskit(label), title='Best partition sampled by simulated QAOA')
plt.show()
# Shared aliases for the optional IQM hardware section below.
opt_circuit = qaoa
opt_counts = counts
exact_score = Q_exact


In [ ]:
# Final comparison: the same two-community modularity objective
score_fn = lambda bits: binary_modularity(bits, B, two_m)
mean_q = sum(score_fn(bits_from_qiskit(label)) * frequency
             for label, frequency in counts.items()) / sum(counts.values())
print(f'Exact two-community optimum: {Q_exact:.4f}')
print(f'QAOA simulated mean:        {mean_q:.4f}')
print(f'QAOA best sampled:         {Q_sample:.4f}')
print(f'Greedy classical Q:        {Q_greedy:.4f} ({len(communities)} communities; may not be directly comparable)')
fig, ax = plt.subplots(figsize=(7.8, 3.5))
ax.bar(['QAOA mean', 'QAOA best', 'Exact optimum'],
       [mean_q, Q_sample, Q_exact], color=['#777777', '#c3212d', '#111111'])
ax.set_ylabel('Modularity Q')
ax.set_title('Two-community modularity: simulator vs exact result', loc='left')
ax.axhline(0, color='#999999', linewidth=.8)
ax.spines[['right', 'top']].set_visible(False)
plt.tight_layout()
plt.show()

## 9 / Interpret your result

The **optimal partition** is the maximum-modularity answer for this small, synthetic, two-community problem. QAOA gives a **distribution** of candidate assignments, not a guaranteed solution.

Check whether QAOA's best sampled solution matches the exact optimum, and whether its **average modularity** is lower. These are different performance measures.

**Possible extensions:** perturb the weights, change the number of nodes, or compare with a classical method that is also restricted to two communities. Real brain connectomes would require additional data validation and neuroscientific interpretation.
**Vocabulary when reading the output:** A *bitstring* is a proposed partition; a *shot* is one measurement; the *best sampled modularity* is the highest-scoring observed partition; the *mean modularity* describes the whole measurement distribution. A perfect best sample does not imply that most samples were good.


---
## Optional / Real IQM hardware (local VS Code only)

The complete simulator exercise above works in Binder **without a token**. To use the same optimized circuit on real IQM hardware, open the notebook locally in VS Code.

1. Use Python 3.11 or 3.12 and install `requirements-hardware.txt`.
2. Run the simulation cells above.
3. Set `RUN_ON_IQM = True` and choose the processor, for example `IQM_QUANTUM_COMPUTER = 'garnet'`.
4. Enter your token through the hidden prompt and review transpilation. Set `CONFIRM_QPU_SUBMISSION = True` only to submit a potentially billable job.

**Never enter private tokens in public Binder.**

In [ ]:
# Stage B / explicit hardware opt-in. No token is used while this is False.
import os
RUN_ON_IQM = False
CONFIRM_QPU_SUBMISSION = False
QPU_SHOTS = 256
IQM_SERVER_URL = os.getenv('IQM_SERVER_URL', 'https://resonance.iqm.tech')
IQM_QUANTUM_COMPUTER = os.getenv('IQM_QUANTUM_COMPUTER', '')

print('IQM hardware stage:', 'ENABLED' if RUN_ON_IQM else 'OFF — simulation complete')


In [ ]:
# Stage C / hidden personal token + backend discovery. Run in trusted Jupyter only.
# Do not enable RUN_ON_IQM in a public mybinder.org session.
from getpass import getpass
from qiskit import transpile

qpu_backend = None
qpu_circuit = None
if RUN_ON_IQM:
    # This is a precaution; always check the real hosting environment yourself.
    if 'mybinder' in str(os.getenv('JUPYTERHUB_HOST', '')).lower() or os.getenv('BINDER_LAUNCH_HOST'):
        raise RuntimeError('Do not provide personal IQM credentials to public Binder. Use local VS Code or trusted Jupyter.')
    try:
        from iqm.qiskit_iqm import IQMProvider
    except ImportError as exc:
        raise ImportError('Install the IQM adapter in this trusted environment: pip install -r requirements-hardware.txt') from exc
    qpu_name = IQM_QUANTUM_COMPUTER.strip() or input('IQM processor name (e.g., garnet): ').strip()
    if not qpu_name:
        raise ValueError('Processor name required. Check IQM Resonance or ask the facilitators.')
    personal_token = os.environ.get('IQM_TOKEN') or getpass('Your IQM API token (input hidden): ')
    if not personal_token:
        raise ValueError('Your IQM token is required for hardware access.')
    try:
        provider = IQMProvider(IQM_SERVER_URL, quantum_computer=qpu_name, token=personal_token)
        qpu_backend = provider.get_backend()
    finally:
        del personal_token  # Never print credentials or save them to the notebook.
    qpu_circuit = transpile(qaoa, backend=qpu_backend, optimization_level=1)
    print('Connected to:', qpu_name, '| physical qubits:', qpu_backend.num_qubits)
    print('Logical depth:', qaoa.depth(), '| transpiled depth:', qpu_circuit.depth())
    print('Mapped operations:', dict(qpu_circuit.count_ops()))
else:
    print('Skipped — no IQM token requested. You can stop after simulation.')


In [ ]:
# Stage D / submit ONLY after manually setting both switches to True.
qpu_counts = None
if RUN_ON_IQM and CONFIRM_QPU_SUBMISSION and qpu_circuit is not None:
    job = qpu_backend.run(qpu_circuit, shots=QPU_SHOTS)
    print('Submitted IQM job:', job.job_id())
    qpu_counts = job.result().get_counts()
    print('Hardware shots collected:', sum(qpu_counts.values()))
else:
    print('No hardware job submitted. Set CONFIRM_QPU_SUBMISSION=True if you are ready.')


In [ ]:
# Stage E / compare classical optimum, simulator and the QPU on the SAME objective.
score_fn = lambda bits: binary_modularity(bits, B, two_m)
simulated_counts = counts

def average_objective(samples):
    total = sum(samples.values())
    return sum(score_fn(bits_from_qiskit(bitstring)) * frequency
               for bitstring, frequency in samples.items()) / total

print('Objective: Modularity')
print('Classical optimum:', round(Q_exact, 6))
print('Simulated mean:', round(average_objective(simulated_counts), 6))
if qpu_counts:
    print('Real IQM mean:', round(average_objective(qpu_counts), 6))
    value, best_bitstring, _ = best_from_counts(qpu_counts, score_fn)
    print('Best observed real-QPU value:', round(value, 6), 'bitstring:', best_bitstring)
else:
    print('Real IQM: not run — simulation-only experiment completed.')


## Reading the final values

- **Exact $Q$:** best two-community score, calculated by checking every assignment.
- **QAOA simulated mean $Q$:** average objective value over simulated measurements.
- **Best sampled $Q$:** highest-scoring observed bitstring, not the average.
- **IQM mean $Q$:** obtained only if you explicitly submit a physical hardware job.

All values use the same network and modularity objective, except the **greedy classical** baseline, which may produce a different number of communities. Finding the optimum in a small example does **not** demonstrate quantum advantage.

## References and provenance

This is a **custom QSIP educational demonstration** built on synthetic data and a custom Qiskit helper module. It is not an official IQM neuroscience notebook or a reproduction of published brain results.

1. Newman, M. E. J. (2006). *Modularity and community structure in networks.* *PNAS*, 103(23), 8577–8582. [doi:10.1073/pnas.0601602103](https://doi.org/10.1073/pnas.0601602103) — modularity objective.
2. Blondel, V. D. et al. (2008). *Fast unfolding of communities in large networks.* *Journal of Statistical Mechanics*, P10008. [doi:10.1088/1742-5468/2008/10/P10008](https://doi.org/10.1088/1742-5468/2008/10/P10008) — classical Louvain approach (background).
3. Farhi, E., Goldstone, J. & Gutmann, S. (2014). *A Quantum Approximate Optimization Algorithm.* [arXiv:1411.4028](https://arxiv.org/abs/1411.4028) — QAOA formulation.
4. [IQM Quantum Computers — Qiskit hardware guide](https://docs.iqm.tech/iqm-client/user_guide_qiskit.html) — real backend execution.

**Limitations:** Modularity depends on the null model and suffers from resolution effects; hardware execution is exploratory and does not imply clinical utility or quantum speedup.

## Optional extension — QAOA parameters trained on hardware

The notebook's default classical optimizer searches for the best QAOA angles $\gamma$ and $\beta$ **in an ideal simulator**. Optional real IQM execution **samples that optimized circuit**. Training QAOA *on hardware* instead would require repeatedly running the QPU while changing the angles, using measured scores as feedback.

That experiment is more advanced and may use significant QPU credits. For an official approach, see [IQM — Training the QAOA](https://docs.iqm.tech/4.6/iqm-qaoa/Training%20the%20QAOA.html). The [IQM Sparse Max-Cut example](https://docs.iqm.tech/4.6/iqm-qaoa/Sparse%20Maxcut.html) illustrates related optimization on hardware.

**Never submit personal IQM credentials to public Binder.**
